# Titanic - Kaggle Notebook

نفس منهجية السكريبتات: imputation جوه الـ Pipeline، الاختيار بالـ CV فقط،
والـ test/submission مرة واحدة في الآخر. يعمل على Kaggle (`/kaggle/input/titanic/`)
أو محليًا على `titanic.csv` (نسخة seaborn).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42

In [ ]:
if Path('/kaggle/input/titanic/train.csv').exists():
    train = pd.read_csv('/kaggle/input/titanic/train.csv')
    test = pd.read_csv('/kaggle/input/titanic/test.csv')
    IS_KAGGLE = True
elif Path('train.csv').exists() and Path('test.csv').exists():
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
    IS_KAGGLE = False  # ملفات كاجل محلية، مش بيئة Kaggle
else:
    train = pd.read_csv('titanic.csv')
    test = None
    IS_KAGGLE = False

print('IS_KAGGLE:', IS_KAGGLE, '| train:', train.shape,
      '| test:', None if test is None else test.shape)
print('target rate:', round(train['Survived' if IS_KAGGLE else 'survived'].mean(), 4))

In [ ]:
def extract_title(name):
    # 'the Countess.' -> Countess (البادئة the اختيارية)
    m = re.search(r',\s*(?:the\s+)?(\w+)\.', str(name))
    return m.group(1) if m else 'Unknown'

print(extract_title('Rothes, the Countess. of (Lucy)'))  # Countess
print(extract_title('Braund, Mr. Owen Harris'))  # Mr

In [ ]:
def clean(df):
    """تنظيف هيكلي فقط: بلا imputation (ده جوه الـ Pipeline)."""
    df = df.copy()
    if 'Name' in df.columns:  # كاجل فقط
        df['Title'] = df['Name'].apply(extract_title).replace({
            'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
            'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer',
            'Dr': 'Officer', 'Rev': 'Officer', 'Don': 'Royalty',
            'Sir': 'Royalty', 'Lady': 'Royalty', 'Countess': 'Royalty',
            'Jonkheer': 'Royalty', 'Dona': 'Royalty'})
    if 'Cabin' in df.columns:
        df['Deck'] = df['Cabin'].str[0].fillna('M')
    df = df.drop(columns=[c for c in ['Cabin', 'Name', 'Ticket', 'deck']
                            if c in df.columns])
    rename = {'Survived': 'survived', 'Pclass': 'pclass', 'Sex': 'sex',
              'Age': 'age', 'SibSp': 'sibsp', 'Parch': 'parch',
              'Fare': 'fare', 'Embarked': 'embarked'}
    return df.rename(columns={k: v for k, v in rename.items()
                                 if k in df.columns})

train_c = clean(train)
test_c = clean(test) if test is not None else None
print(train_c.shape, list(train_c.columns))

In [ ]:
def add_features(X):
    X = X.copy()
    X['family_size'] = X['sibsp'] + X['parch'] + 1
    X['alone'] = (X['family_size'] == 1).astype(int)
    X['age_missing'] = X['age'].isna().astype(int)
    X['is_child'] = (X['age'] < 12).astype(int)
    X['fare_per_person'] = X['fare'] / X['family_size']
    X['age_x_pclass'] = X['age'] * X['pclass']
    return X

BASE = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
ENG = ['family_size', 'alone', 'age_missing', 'is_child',
       'fare_per_person', 'age_x_pclass']
EXTRA = ['Title', 'Deck']  # كاجل فقط — تُضاف لو موجودة
FEATS = BASE + ENG + [c for c in EXTRA if c in train_c.columns]

train_f = add_features(train_c)
# pandas 3: الأعمدة النصية dtype=str لا object — لذلك select_dtypes
ALL_NUM = train_f[FEATS].select_dtypes(include='number').columns.tolist()
num_cols = [c for c in ALL_NUM if c != 'pclass']  # pclass فئوية مش رقمية
cat_cols = [c for c in FEATS if c not in num_cols]
print('num:', num_cols, '\ncat:', cat_cols)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train_c.groupby('sex')['survived'].mean().plot(
    kind='bar', ax=axes[0], title='Survival by sex')
train_c.groupby('pclass')['survived'].mean().plot(
    kind='bar', ax=axes[1], title='Survival by pclass')
plt.tight_layout()
plt.show()

In [ ]:
X = train_f[FEATS]
y = train_f['survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('oh', OneHotEncoder(handle_unknown='ignore',
                                                drop='if_binary'))]), cat_cols),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# baseline: أغلبية + قاعدة الجنس
for name, clf in [('Dummy', DummyClassifier(strategy='most_frequent'))]:
    clf.fit(X_train, y_train)
    print(name, round(accuracy_score(y_test, clf.predict(X_test)), 4))
sex_acc = (((X_test['sex'] == 'female').astype(int) == y_test).mean())
print('SexRule:', round(float(sex_acc), 4))

In [ ]:
grids = {
    'LR': (LogisticRegression(max_iter=1000),
           {'clf__C': [0.05, 0.1, 0.5, 1.0]}),
    'RF': (RandomForestClassifier(n_estimators=300, random_state=SEED),
           {'clf__max_depth': [6, 8, None], 'clf__min_samples_split': [2, 5]}),
    'HGB': (HistGradientBoostingClassifier(random_state=SEED),
            {'clf__learning_rate': [0.05, 0.1],
             'clf__max_leaf_nodes': [15, 31]}),
}

best_score, best_gs, best_name = 0, None, ''
for name, (clf, grid) in grids.items():
    gs = GridSearchCV(Pipeline([('prep', prep), ('clf', clf)]),
                      grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    gs.fit(X_train, y_train)
    print(name, gs.best_params_, round(gs.best_score_, 4))
    if gs.best_score_ > best_score:
        best_score, best_gs, best_name = gs.best_score_, gs, name
print('Best by CV:', best_name)

In [ ]:
# الـ test مرة واحدة للتقارير فقط
pred = best_gs.predict(X_test)
print('test_acc:', round(accuracy_score(y_test, pred), 4),
      '| test_AUC:',
      round(roc_auc_score(y_test, best_gs.predict_proba(X_test)[:, 1]), 4))

In [ ]:
# submission: refit على كامل الـ train (test.csv منفصل — لا تسريب)
if test_c is not None:
    final = best_gs.best_estimator_.fit(X, y)
    preds = final.predict(add_features(test_c)[FEATS])
    pid = test_c['PassengerId'] if 'PassengerId' in test_c.columns else test_c.index
    sub = pd.DataFrame({'PassengerId': pid, 'Survived': preds})
    sub.to_csv('submission.csv', index=False)
    print(sub.shape)
    sub.head()
else:
    print('No test.csv: practicing on local split only.')